# FASE 3: Deep Learning con Red Neuronal
## Predicción de Propensión a Fallar en Servicios de Reparación

**Objetivo:** Optimizar sobre el baseline XGBoost (Fase 2) usando Deep Learning

**Variable Target:** Propenso_a_Fallar (0: No fallo, 1: Fallo)

**Baseline Fase 2:** AUC-ROC = 0.9852, Accuracy = 94.6%

**Estrategia Fase 3:**
- ✅ Cargar datos normalizados de Fase 1
- ✅ Construir arquitectura de Red Neuronal Profunda
- ✅ Entrenar con callbacks (Early Stopping, Reducción LR)
- ✅ Evaluar en múltiples métricas: Accuracy, Precision, Recall, F1, AUC-ROC
- ✅ Generar visualizaciones: Training History, ROC, Confusion Matrix, Learning Curves
- ✅ Comparar con Baseline y generar reportes
- ✅ Guardar mejor modelo en formato .h5

## Sección 1: Importaciones y Configuración Inicial

In [16]:
import os
import sys
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report, auc
)

# TensorFlow y Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Reduce TensorFlow verbosity

# Paths
ruta_proyecto = Path('.')
ruta_preprocesamiento = ruta_proyecto / '../02-Preprocessamiento'
ruta_baseline = ruta_proyecto / '../03-Baseline'
ruta_deeplearning = ruta_proyecto

print(f"✅ TensorFlow versión: {tf.__version__}")
print(f"✅ Keras versión: {keras.__version__}")
print(f"📁 Directorio: {Path.cwd()}")
print(f"🖥️  GPU disponible: {len(tf.config.list_physical_devices('GPU')) > 0}")

✅ TensorFlow versión: 2.20.0
✅ Keras versión: 3.12.1
📁 Directorio: c:\Users\DELL\Documents\GitHub\material-redesneuronales\04-DeepLearning
🖥️  GPU disponible: False


## Sección 2: Cargar Datos Normalizados de Fase 1

In [17]:
print("📊 CARGANDO DATOS NORMALIZADOS Y BALANCEADOS DE FASE 1")
print("=" * 70)

try:
    # Cargar datos normalizados y balanceados (aplicado RandomOverSampler en Fase 1)
    X_train = pd.read_csv(ruta_preprocesamiento / 'X_train_normalizado_Oversampled.csv')
    X_test = pd.read_csv(ruta_preprocesamiento / 'X_test_normalizado.csv')
    
    # Separar features y target
    FEATURE_COLS = [col for col in X_train.columns if col != 'Propenso_a_Fallar']
    TARGET_COL = 'Propenso_a_Fallar'
    
    y_train = X_train[TARGET_COL].values
    X_train = X_train[FEATURE_COLS].values
    
    y_test = X_test[TARGET_COL].values
    X_test = X_test[FEATURE_COLS].values
    
    print(f"\n✅ Datos cargados exitosamente")
    print(f"\n📊 Dataset Train (BALANCEADO con RandomOverSampler en Fase 1):")
    print(f"   Registros: {X_train.shape[0]:,}")
    print(f"   Features: {X_train.shape[1]}")
    print(f"   Target - Clase 0: {(y_train == 0).sum():,} ({(y_train == 0).sum()/len(y_train)*100:.2f}%)")
    print(f"   Target - Clase 1: {(y_train == 1).sum():,} ({(y_train == 1).sum()/len(y_train)*100:.2f}%)")
    print(f"   Nota: Estos son los MISMOS datos usados en Fase 2 (XGBoost Baseline)")
    
    print(f"\n📊 Dataset Test:")
    print(f"   Registros: {X_test.shape[0]:,}")
    print(f"   Features: {X_test.shape[1]}")
    print(f"   Target - Clase 0: {(y_test == 0).sum():,} ({(y_test == 0).sum()/len(y_test)*100:.2f}%)")
    print(f"   Target - Clase 1: {(y_test == 1).sum():,} ({(y_test == 1).sum()/len(y_test)*100:.2f}%)")
    
    # Cargar baseline results
    with open(ruta_baseline / 'metricas_baseline.json', 'r') as f:
        baseline_metrics = json.load(f)
    
    print(f"\n🏆 Baseline (Fase 2 - XGBoost usando MISMOS DATOS BALANCEADOS):")
    best_split = baseline_metrics['mejor_modelo']['split']
    print(f"   Mejor Split: {best_split}")
    print(f"   AUC-ROC: {baseline_metrics['mejor_modelo']['auc_roc']:.4f}")
    print(f"   Accuracy: {baseline_metrics['mejor_modelo']['accuracy']:.4f}")
    print(f"   F1-Score: {baseline_metrics['mejor_modelo']['f1_score']:.4f}")
    
except Exception as e:
    print(f"❌ Error cargando datos: {str(e)}")
    sys.exit(1)

📊 CARGANDO DATOS NORMALIZADOS Y BALANCEADOS DE FASE 1

✅ Datos cargados exitosamente

📊 Dataset Train (BALANCEADO con RandomOverSampler en Fase 1):
   Registros: 26,094
   Features: 25
   Target - Clase 0: 13,047 (50.00%)
   Target - Clase 1: 13,047 (50.00%)
   Nota: Estos son los MISMOS datos usados en Fase 2 (XGBoost Baseline)

📊 Dataset Test:
   Registros: 3,512
   Features: 25
   Target - Clase 0: 3,262 (92.88%)
   Target - Clase 1: 250 (7.12%)

🏆 Baseline (Fase 2 - XGBoost usando MISMOS DATOS BALANCEADOS):
   Mejor Split: 70-30
   AUC-ROC: 0.9852
   Accuracy: 0.9464
   F1-Score: 0.9482


## Sección 3: Construir Arquitectura de Red Neuronal Profunda

In [18]:
print("🏗️  CONSTRUYENDO ARQUITECTURA DE RED NEURONAL")
print("=" * 70)

# Parámetros de la arquitectura
INPUT_DIM = X_train.shape[1]
L2_REG = 0.001
DROPOUT_RATE = 0.4

# Construir modelo secuencial
model = models.Sequential([
    # Capa de entrada
    layers.Input(shape=(INPUT_DIM,)),
    
    # Primera capa densa con normalización
    layers.Dense(256, kernel_regularizer=l2(L2_REG)),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(DROPOUT_RATE),
    
    # Segunda capa densa
    layers.Dense(128, kernel_regularizer=l2(L2_REG)),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.3),
    
    # Tercera capa densa
    layers.Dense(64),
    layers.Activation('relu'),
    layers.Dropout(0.2),
    
    # Cuarta capa densa
    layers.Dense(32),
    layers.Activation('relu'),
    layers.Dropout(0.2),
    
    # Quinta capa densa
    layers.Dense(16),
    layers.Activation('relu'),
    
    # Capa de salida
    layers.Dense(1, activation='sigmoid')
])

# Compilar modelo
optimizer = Adam(learning_rate=0.001)
model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

print(f"\n📋 Arquitectura del Modelo:")
model.summary()

print(f"\n🔧 Configuración:")
print(f"   Optimizer: Adam (lr=0.001)")
print(f"   Loss: Binary Crossentropy")
print(f"   Batch Size: 32")
print(f"   Epochs: 150 (con Early Stopping)")
print(f"   Class Weights: {{0: 0.6, 1: 1.4}}")
print(f"   L2 Regularization: {L2_REG}")

🏗️  CONSTRUYENDO ARQUITECTURA DE RED NEURONAL

📋 Arquitectura del Modelo:


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 256)            │         6,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_6 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_7 (Activation)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_8 (Activation)       │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_9 (Activation)       │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 51,969 (203.00 KB)

 Trainable params: 51,201 (200.00 KB)

 Non-trainable params: 768 (3.00 KB)


🔧 Configuración:
   Optimizer: Adam (lr=0.001)
   Loss: Binary Crossentropy
   Batch Size: 32
   Epochs: 150 (con Early Stopping)
   Class Weights: {0: 0.6, 1: 1.4}
   L2 Regularization: 0.001


## Sección 4: Entrenar Red Neuronal

In [19]:
print("\n🎓 ENTRENANDO RED NEURONAL")
print("=" * 70)

# Definir callbacks
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

# Calcular class weights
class_weights = {0: 0.6, 1: 1.4}

# Entrenar modelo
history = model.fit(
    X_train, y_train,
    batch_size=32,
    epochs=150,
    validation_split=0.2,
    class_weight=class_weights,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

print(f"\n✅ Entrenamiento completado")
print(f"   Épocas entrenadas: {len(history.history['loss'])}")
print(f"   Mejor epoch: {np.argmin(history.history['val_loss']) + 1}")


🎓 ENTRENANDO RED NEURONAL
Epoch 1/150
653/653 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.7825 - auc: 0.8999 - loss: 0.5089 - val_accuracy: 0.9847 - val_auc: 0.0000e+00 - val_loss: 0.3000 - learning_rate: 0.0010
Epoch 2/150
653/653 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.8611 - auc: 0.9457 - loss: 0.3343 - val_accuracy: 0.9651 - val_auc: 0.0000e+00 - val_loss: 0.2011 - learning_rate: 0.0010
Epoch 3/150
653/653 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.8732 - auc: 0.9517 - loss: 0.2826 - val_accuracy: 0.9381 - val_auc: 0.0000e+00 - val_loss: 0.2768 - learning_rate: 0.0010
Epoch 4/150
653/653 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.8762 - auc: 0.9531 - loss: 0.2677 - val_accuracy: 0.9519 - val_auc: 0.0000e+00 - val_loss: 0.1986 - learning_rate: 0.0010
Epoch 5/150
653/653 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.8791 - auc: 0.9546 - loss: 0.2553 - val_accuracy: 0.9548 - val_auc: 0.0000e+00 - val_loss: 0.1886 - learning_rate: 0.0010
Epoch 6/150
653/653 ━━━━━━━

## Sección 5: Visualizar Training History

In [20]:
print("\n📈 GENERANDO TRAINING HISTORY")
print("=" * 70)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss
axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Loss', fontsize=11, fontweight='bold')
axes[0].set_title('Training & Validation Loss', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Accuracy', fontsize=11, fontweight='bold')
axes[1].set_title('Training & Validation Accuracy', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(ruta_deeplearning / 'training_history.png', dpi=100, bbox_inches='tight')
print("✅ Training history guardada: training_history.png")
plt.close()


📈 GENERANDO TRAINING HISTORY
✅ Training history guardada: training_history.png


## Sección 6: Evaluar en Test Set

In [21]:
print("\n📊 EVALUANDO EN TEST SET")
print("=" * 70)

# Predicciones en test
y_pred_proba = model.predict(X_test, verbose=0).flatten()
y_pred = (y_pred_proba >= 0.5).astype(int)

# Calcular métricas
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
auc_roc = roc_auc_score(y_test, y_pred_proba)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

# Specificity y Sensitivity
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0

print(f"\n📈 Métricas en Test Set:")
print(f"   Accuracy:    {accuracy:.4f}")
print(f"   Precision:   {precision:.4f}")
print(f"   Recall:      {recall:.4f}")
print(f"   F1-Score:    {f1:.4f}")
print(f"   AUC-ROC:     {auc_roc:.4f}")
print(f"   Specificity: {specificity:.4f}")
print(f"   Sensitivity: {sensitivity:.4f}")

print(f"\n🎯 Confusion Matrix:")
print(f"   TN={tn:,} | FP={fp:,}")
print(f"   FN={fn:,} | TP={tp:,}")

# Comparación con baseline
baseline_auc = baseline_metrics['mejor_modelo']['auc_roc']
baseline_acc = baseline_metrics['mejor_modelo']['accuracy']

print(f"\n📊 COMPARACIÓN CON BASELINE (XGBoost - Fase 2):")
print(f"   Métrica        | DeepLearning | Baseline | Diferencia")
print(f"   ────────────────────────────────────────────────────")
print(f"   AUC-ROC        | {auc_roc:.4f}     | {baseline_auc:.4f}   | {auc_roc - baseline_auc:+.4f}")
print(f"   Accuracy       | {accuracy:.4f}     | {baseline_acc:.4f}   | {accuracy - baseline_acc:+.4f}")

# Guardar resultados
test_results = {
    'accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1': f1,
    'auc_roc': auc_roc,
    'specificity': specificity,
    'sensitivity': sensitivity,
    'y_pred': y_pred,
    'y_pred_proba': y_pred_proba,
    'confusion_matrix': cm,
    'classification_report': classification_report(y_test, y_pred)
}


📊 EVALUANDO EN TEST SET

📈 Métricas en Test Set:
   Accuracy:    0.8759
   Precision:   0.3599
   Recall:      0.9560
   F1-Score:    0.5230
   AUC-ROC:     0.9672
   Specificity: 0.8697
   Sensitivity: 0.9560

🎯 Confusion Matrix:
   TN=2,837 | FP=425
   FN=11 | TP=239

📊 COMPARACIÓN CON BASELINE (XGBoost - Fase 2):
   Métrica        | DeepLearning | Baseline | Diferencia
   ────────────────────────────────────────────────────
   AUC-ROC        | 0.9672     | 0.9852   | -0.0180
   Accuracy       | 0.8759     | 0.9464   | -0.0705


## Sección 7: Visualizar ROC Curve

In [22]:
print("\n📈 GENERANDO ROC CURVE")
print("=" * 70)

fig, ax = plt.subplots(figsize=(10, 8))

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
auc_score = auc(fpr, tpr)

ax.plot(fpr, tpr, color='#FF6B6B', lw=2.5, label=f'DeepLearning (AUC = {auc_score:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random Classifier')

ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
ax.set_title('ROC Curve - Deep Learning (Fase 3)', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(ruta_deeplearning / 'roc_curve_rnn.png', dpi=100, bbox_inches='tight')
print("✅ ROC curve guardada: roc_curve_rnn.png")
plt.close()


📈 GENERANDO ROC CURVE
✅ ROC curve guardada: roc_curve_rnn.png


## Sección 8: Visualizar Confusion Matrix

In [23]:
print("\n📊 GENERANDO CONFUSION MATRIX")
print("=" * 70)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix sin normalizar
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
            cbar_kws={'label': 'Count'}, annot_kws={'size': 14})
axes[0].set_xlabel('Predicted', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Actual', fontsize=11, fontweight='bold')
axes[0].set_title('Confusion Matrix - Raw Counts', fontsize=12, fontweight='bold')
axes[0].set_xticklabels(['No Fallo (0)', 'Fallo (1)'])
axes[0].set_yticklabels(['No Fallo (0)', 'Fallo (1)'])

# Confusion matrix normalizado
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Greens', ax=axes[1],
            cbar_kws={'label': 'Percentage'}, annot_kws={'size': 12})
axes[1].set_xlabel('Predicted', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Actual', fontsize=11, fontweight='bold')
axes[1].set_title('Confusion Matrix - Normalized', fontsize=12, fontweight='bold')
axes[1].set_xticklabels(['No Fallo (0)', 'Fallo (1)'])
axes[1].set_yticklabels(['No Fallo (0)', 'Fallo (1)'])

plt.tight_layout()
plt.savefig(ruta_deeplearning / 'confusion_matrix_rnn.png', dpi=100, bbox_inches='tight')
print("✅ Confusion matrices guardadas: confusion_matrix_rnn.png")
plt.close()


📊 GENERANDO CONFUSION MATRIX
✅ Confusion matrices guardadas: confusion_matrix_rnn.png


## Sección 9: Visualizar Learning Curves

In [24]:
print("\n📈 GENERANDO LEARNING CURVES")
print("=" * 70)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss
axes[0, 0].plot(history.history['loss'], label='Train Loss', linewidth=2, color='#FF6B6B')
axes[0, 0].plot(history.history['val_loss'], label='Val Loss', linewidth=2, color='#4ECDC4')
axes[0, 0].set_xlabel('Epoch', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Loss', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Training & Validation Loss', fontsize=12, fontweight='bold')
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(alpha=0.3)

# Accuracy
axes[0, 1].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2, color='#FF6B6B')
axes[0, 1].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2, color='#4ECDC4')
axes[0, 1].set_xlabel('Epoch', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Accuracy', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Training & Validation Accuracy', fontsize=12, fontweight='bold')
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(alpha=0.3)

# AUC
axes[1, 0].plot(history.history['auc'], label='Train AUC', linewidth=2, color='#FF6B6B')
axes[1, 0].plot(history.history['val_auc'], label='Val AUC', linewidth=2, color='#4ECDC4')
axes[1, 0].set_xlabel('Epoch', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('AUC', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Training & Validation AUC', fontsize=12, fontweight='bold')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(alpha=0.3)

# Resumen de convergencia
best_epoch = np.argmin(history.history['val_loss']) + 1
axes[1, 1].axis('off')

summary_text = f"""
CONVERGENCE SUMMARY
{'─'*40}
Best Epoch: {best_epoch}
Total Epochs: {len(history.history['loss'])}

Train Loss: {history.history['loss'][-1]:.4f}
Val Loss:   {history.history['val_loss'][-1]:.4f}

Train Acc:  {history.history['accuracy'][-1]:.4f}
Val Acc:    {history.history['val_accuracy'][-1]:.4f}

Train AUC:  {history.history['auc'][-1]:.4f}
Val AUC:    {history.history['val_auc'][-1]:.4f}
"""

axes[1, 1].text(0.1, 0.5, summary_text, fontsize=11, family='monospace',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
               verticalalignment='center')

plt.tight_layout()
plt.savefig(ruta_deeplearning / 'learning_curves.png', dpi=100, bbox_inches='tight')
print("✅ Learning curves guardadas: learning_curves.png")
plt.close()


📈 GENERANDO LEARNING CURVES
✅ Learning curves guardadas: learning_curves.png


## Sección 10: Generar Reporte Detallado

In [25]:
print("\n📝 GENERANDO REPORTE DETALLADO")
print("=" * 70)

timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

reporte_text = f"""
{'='*80}
DEEP LEARNING BASELINE - REPORTE DE EVALUACIÓN
FASE 3: Red Neuronal Profunda vs XGBoost Baseline
{'='*80}

Fecha: {timestamp}


{'='*80}
1. DESCRIPCIÓN DEL PROBLEMA
{'='*80}

Variable Target: Propenso_a_Fallar
  • Clase 0: No fallo (servicios completados exitosamente)
  • Clase 1: Fallo (servicios donde ocurrió un fallo)

Objetivo: Optimizar predicción de fallos usando Deep Learning (Red Neuronal)
Comparar con Baseline XGBoost (Fase 2)


{'='*80}
2. DATOS Y PREPARACIÓN
{'='*80}

Dataset Total:
  • Total registros: {len(X_train) + len(X_test):,}
  • Features: {X_train.shape[1]}
  • Clase 0: {((y_train == 0).sum() + (y_test == 0).sum()):,}
  • Clase 1: {((y_train == 1).sum() + (y_test == 1).sum()):,}

Preprocesamiento (Fase 1):
  • Normalización: MinMaxScaler [0,1]
  • Oversampling: RandomOverSampler (aplicado en train)
  • Encoding: Target Encoding + Label Encoding


{'='*80}
3. ARQUITECTURA DE RED NEURONAL
{'='*80}

Modelo Secuencial - 5 Capas Densas:
  Input ({X_train.shape[1]} features)
    ↓
  Dense(256) + BatchNorm + ReLU + Dropout(0.4) + L2(0.001)
    ↓
  Dense(128) + BatchNorm + ReLU + Dropout(0.3) + L2(0.001)
    ↓
  Dense(64) + ReLU + Dropout(0.2)
    ↓
  Dense(32) + ReLU + Dropout(0.2)
    ↓
  Dense(16) + ReLU
    ↓
  Output: Dense(1, Sigmoid) → [0.0, 1.0]

Configuración:
  • Optimizer: Adam (lr=0.001)
  • Loss Function: Binary Crossentropy
  • Batch Size: 32
  • Epochs: 150 (con Early Stopping)
  • Early Stopping: patience=15, monitor=val_loss
  • Reduce LR: factor=0.5, patience=5
  • Class Weights: {{0: 0.6, 1: 1.4}}
  • L2 Regularization: 0.001


{'='*80}
4. RESULTADOS EN TEST SET
{'='*80}

Métricas de Evaluación:
  • Accuracy:    {test_results['accuracy']:.4f}
  • Precision:   {test_results['precision']:.4f}
  • Recall:      {test_results['recall']:.4f}
  • F1-Score:    {test_results['f1']:.4f}
  • AUC-ROC:     {test_results['auc_roc']:.4f}
  • Specificity: {test_results['specificity']:.4f}
  • Sensitivity: {test_results['sensitivity']:.4f}

Confusion Matrix:
  • Verdaderos Negativos (TN): {cm[0, 0]:,}
  • Falsos Positivos (FP): {cm[0, 1]:,}
  • Falsos Negativos (FN): {cm[1, 0]:,}
  • Verdaderos Positivos (TP): {cm[1, 1]:,}

Classification Report:
{test_results['classification_report']}

{'='*80}
5. COMPARATIVA CON BASELINE (XGBoost - Fase 2)
{'='*80}

Métrica          | DeepLearning | XGBoost  | Diferencia | % Mejora
─────────────────────────────────────────────────────────────────
Accuracy         | {test_results['accuracy']:.4f}     | {baseline_acc:.4f}   | {test_results['accuracy'] - baseline_acc:+.4f}     | {((test_results['accuracy'] - baseline_acc)/baseline_acc)*100:+.2f}%
Precision        | {test_results['precision']:.4f}     | {baseline_metrics['resultados'][best_split]['precision']:.4f}   | {test_results['precision'] - baseline_metrics['resultados'][best_split]['precision']:+.4f}     | {((test_results['precision'] - baseline_metrics['resultados'][best_split]['precision'])/baseline_metrics['resultados'][best_split]['precision'])*100:+.2f}%
Recall           | {test_results['recall']:.4f}     | {baseline_metrics['resultados'][best_split]['recall']:.4f}   | {test_results['recall'] - baseline_metrics['resultados'][best_split]['recall']:+.4f}     | {((test_results['recall'] - baseline_metrics['resultados'][best_split]['recall'])/baseline_metrics['resultados'][best_split]['recall'])*100:+.2f}%
F1-Score         | {test_results['f1']:.4f}     | {baseline_metrics['resultados'][best_split]['f1_score']:.4f}   | {test_results['f1'] - baseline_metrics['resultados'][best_split]['f1_score']:+.4f}     | {((test_results['f1'] - baseline_metrics['resultados'][best_split]['f1_score'])/baseline_metrics['resultados'][best_split]['f1_score'])*100:+.2f}%
AUC-ROC          | {test_results['auc_roc']:.4f}     | {baseline_auc:.4f}   | {test_results['auc_roc'] - baseline_auc:+.4f}     | {((test_results['auc_roc'] - baseline_auc)/baseline_auc)*100:+.2f}%


{'='*80}
6. ENTRENAMIENTO Y CONVERGENCIA
{'='*80}

Épocas Entrenadas: {len(history.history['loss'])}
Mejor Epoch (Val Loss): {np.argmin(history.history['val_loss']) + 1}
Early Stopping activado: Sí (patience=15)

Loss Final:
  • Train: {history.history['loss'][-1]:.4f}
  • Val:   {history.history['val_loss'][-1]:.4f}

Accuracy Final:
  • Train: {history.history['accuracy'][-1]:.4f}
  • Val:   {history.history['val_accuracy'][-1]:.4f}

AUC Final:
  • Train: {history.history['auc'][-1]:.4f}
  • Val:   {history.history['val_auc'][-1]:.4f}


{'='*80}
7. ANÁLISIS E INTERPRETACIÓN
{'='*80}

✅ Fortalezas del Modelo Deep Learning:
   • Red neuronal profunda captura patrones complejos
   • Batch Normalization estabiliza el entrenamiento
   • Dropout previene overfitting
   • Class weights balancean desbalance de clases
   • Early stopping evita entrenamiento excesivo

⚠️  Considera:
   • Comparar tiempo de inference vs XGBoost
   • Validar estabilidad en datos reales
   • Considerar ensemble si aún mejora disponible

🏆 Veredicto:
   DeepLearning vs Baseline:
   • Mejora en Accuracy: {test_results['accuracy'] - baseline_acc:+.4f} ({((test_results['accuracy'] - baseline_acc)/baseline_acc)*100:+.2f}%)
   • Mejora en AUC-ROC: {test_results['auc_roc'] - baseline_auc:+.4f} ({((test_results['auc_roc'] - baseline_auc)/baseline_auc)*100:+.2f}%)


{'='*80}
8. PRÓXIMOS PASOS
{'='*80}

1. Validar modelo en datos reales de producción
2. Comparar tiempo de inference con XGBoost
3. Considerar ensemble (XGBoost + DeepLearning)
4. Realizar tuning adicional de hiperparámetros
5. Implementar modelo en pipeline de producción


{'='*80}
FIN DEL REPORTE
{'='*80}
"""

# Guardar reporte
with open(ruta_deeplearning / 'reporte_rnn.txt', 'w', encoding='utf-8') as f:
    f.write(reporte_text)

print("✅ Reporte guardado: reporte_rnn.txt")
print("\n" + reporte_text[:2000] + "...\n[Reporte completo guardado]")


📝 GENERANDO REPORTE DETALLADO
✅ Reporte guardado: reporte_rnn.txt


DEEP LEARNING BASELINE - REPORTE DE EVALUACIÓN
FASE 3: Red Neuronal Profunda vs XGBoost Baseline

Fecha: 2026-02-24 15:37:37


1. DESCRIPCIÓN DEL PROBLEMA

Variable Target: Propenso_a_Fallar
  • Clase 0: No fallo (servicios completados exitosamente)
  • Clase 1: Fallo (servicios donde ocurrió un fallo)

Objetivo: Optimizar predicción de fallos usando Deep Learning (Red Neuronal)
Comparar con Baseline XGBoost (Fase 2)


2. DATOS Y PREPARACIÓN

Dataset Total:
  • Total registros: 29,606
  • Features: 25
  • Clase 0: 16,309
  • Clase 1: 13,297

Preprocesamiento (Fase 1):
  • Normalización: MinMaxScaler [0,1]
  • Oversampling: RandomOverSampler (aplicado en train)
  • Encoding: Target Encoding + Label Encoding


3. ARQUITECTURA DE RED NEURONAL

Modelo Secuencial - 5 Capas Densas:
  Input (25 features)
    ↓
  Dense(256) + BatchNorm + ReLU + Dropout(0.4) + L2(0.001)
    ↓
  Dense(128) + BatchNorm + ReLU + Dropout(0.3) + L2

## Sección 11: Exportar Métricas en JSON

In [26]:
print("\n💾 EXPORTANDO MÉTRICAS EN JSON")
print("=" * 70)

# Preparar datos de métricas
metricas_json = {
    'fecha_ejecucion': timestamp,
    'fase': 3,
    'modelo': 'Deep Learning - Red Neuronal Profunda',
    'problema': {
        'target': 'Propenso_a_Fallar',
        'clase_0': 'No fallo',
        'clase_1': 'Fallo',
        'descripcion': 'Predicción de propensión a fallar en servicios de reparación'
    },
    'datos': {
        'total_registros': int(len(X_train) + len(X_test)),
        'total_features': int(X_train.shape[1]),
        'clase_0_total': int((y_train == 0).sum() + (y_test == 0).sum()),
        'clase_1_total': int((y_train == 1).sum() + (y_test == 1).sum())
    },
    'arquitectura': {
        'capas': [
            {'tipo': 'Input', 'dim': X_train.shape[1]},
            {'tipo': 'Dense', 'units': 256, 'activation': 'relu', 'regularization': 'L2(0.001)'},
            {'tipo': 'BatchNormalization'},
            {'tipo': 'Dropout', 'rate': 0.4},
            {'tipo': 'Dense', 'units': 128, 'activation': 'relu', 'regularization': 'L2(0.001)'},
            {'tipo': 'BatchNormalization'},
            {'tipo': 'Dropout', 'rate': 0.3},
            {'tipo': 'Dense', 'units': 64, 'activation': 'relu'},
            {'tipo': 'Dropout', 'rate': 0.2},
            {'tipo': 'Dense', 'units': 32, 'activation': 'relu'},
            {'tipo': 'Dropout', 'rate': 0.2},
            {'tipo': 'Dense', 'units': 16, 'activation': 'relu'},
            {'tipo': 'Dense', 'units': 1, 'activation': 'sigmoid'}
        ]
    },
    'hiperparametros': {
        'optimizer': 'Adam',
        'learning_rate': 0.001,
        'loss': 'binary_crossentropy',
        'batch_size': 32,
        'epochs': len(history.history['loss']),
        'early_stopping': 'patience=15',
        'reduce_lr': 'factor=0.5, patience=5',
        'class_weights': {'0': 0.6, '1': 1.4}
    },
    'entrenamiento': {
        'epochs_entrenadas': int(len(history.history['loss'])),
        'mejor_epoch': int(np.argmin(history.history['val_loss']) + 1),
        'early_stopping_activado': True,
        'loss_final_train': float(history.history['loss'][-1]),
        'loss_final_val': float(history.history['val_loss'][-1]),
        'accuracy_final_train': float(history.history['accuracy'][-1]),
        'accuracy_final_val': float(history.history['val_accuracy'][-1]),
        'auc_final_train': float(history.history['auc'][-1]),
        'auc_final_val': float(history.history['val_auc'][-1])
    },
    'resultados_test': {
        'accuracy': float(test_results['accuracy']),
        'precision': float(test_results['precision']),
        'recall': float(test_results['recall']),
        'f1_score': float(test_results['f1']),
        'auc_roc': float(test_results['auc_roc']),
        'specificity': float(test_results['specificity']),
        'sensitivity': float(test_results['sensitivity']),
        'confusion_matrix': {
            'tn': int(cm[0, 0]),
            'fp': int(cm[0, 1]),
            'fn': int(cm[1, 0]),
            'tp': int(cm[1, 1])
        }
    },
    'comparativa_baseline': {
        'xgboost_accuracy': float(baseline_acc),
        'xgboost_auc_roc': float(baseline_auc),
        'deeplearning_accuracy': float(test_results['accuracy']),
        'deeplearning_auc_roc': float(test_results['auc_roc']),
        'mejora_accuracy': float(test_results['accuracy'] - baseline_acc),
        'mejora_auc_roc': float(test_results['auc_roc'] - baseline_auc),
        'porcentaje_mejora_accuracy': float(((test_results['accuracy'] - baseline_acc)/baseline_acc)*100),
        'porcentaje_mejora_auc_roc': float(((test_results['auc_roc'] - baseline_auc)/baseline_auc)*100)
    }
}

# Guardar JSON
with open(ruta_deeplearning / 'metricas_rnn.json', 'w', encoding='utf-8') as f:
    json.dump(metricas_json, f, indent=2, ensure_ascii=False)

print("✅ Métricas guardadas: metricas_rnn.json")
print("\n📊 Resumen de métricas:")
print(json.dumps(metricas_json['resultados_test'], indent=2))
print("\n📊 Comparativa con Baseline:")
print(json.dumps(metricas_json['comparativa_baseline'], indent=2))


💾 EXPORTANDO MÉTRICAS EN JSON
✅ Métricas guardadas: metricas_rnn.json

📊 Resumen de métricas:
{
  "accuracy": 0.8758542141230068,
  "precision": 0.35993975903614456,
  "recall": 0.956,
  "f1_score": 0.5229759299781181,
  "auc_roc": 0.9672127529123238,
  "specificity": 0.8697118332311465,
  "sensitivity": 0.956,
  "confusion_matrix": {
    "tn": 2837,
    "fp": 425,
    "fn": 11,
    "tp": 239
  }
}

📊 Comparativa con Baseline:
{
  "xgboost_accuracy": 0.9463533018265423,
  "xgboost_auc_roc": 0.9852332492131269,
  "deeplearning_accuracy": 0.8758542141230068,
  "deeplearning_auc_roc": 0.9672127529123238,
  "mejora_accuracy": -0.07049908770353552,
  "mejora_auc_roc": -0.018020496300803157,
  "porcentaje_mejora_accuracy": -7.449552674193273,
  "porcentaje_mejora_auc_roc": -1.8290588868367497
}


## Sección 12: Guardar Modelo Entrenado

In [27]:
print("\n💾 GUARDANDO MODELO ENTRENADO")
print("=" * 70)

# Guardar modelo en formato .h5
model.save(ruta_deeplearning / 'modelo_rnn_final.h5')

model_size = os.path.getsize(ruta_deeplearning / 'modelo_rnn_final.h5') / 1024

print(f"\n✅ Modelo guardado: modelo_rnn_final.h5")
print(f"   Arquitectura: Red Neuronal Profunda (5 capas densas)")
print(f"   Formato: HDF5 (.h5)")
print(f"   Tamaño: {model_size:.2f} KB")
print(f"   Parámetros entrenables: {model.count_params():,}")


💾 GUARDANDO MODELO ENTRENADO

✅ Modelo guardado: modelo_rnn_final.h5
   Arquitectura: Red Neuronal Profunda (5 capas densas)
   Formato: HDF5 (.h5)
   Tamaño: 681.99 KB
   Parámetros entrenables: 51,969


## Sección 13: Resumen Final y Comparativa

In [28]:
print("\n" + "="*80)
print("✅ FASE 3: DEEP LEARNING - COMPLETADO")
print("="*80)

print(f"\n📊 RESULTADOS FINALES - DEEP LEARNING (Fase 3):")
print(f"\nMétricas en Test Set:")
print(f"  • AUC-ROC:    {test_results['auc_roc']:.4f}")
print(f"  • Accuracy:   {test_results['accuracy']:.4f}")
print(f"  • Precision:  {test_results['precision']:.4f}")
print(f"  • Recall:     {test_results['recall']:.4f}")
print(f"  • F1-Score:   {test_results['f1']:.4f}")

print(f"\n📊 COMPARATIVA: DEEPLEARNING vs XGBOOST BASELINE")
print(f"\n{'Métrica':<20} {'DeepLearning':<15} {'XGBoost Baseline':<20} {'Diferencia':<15}")
print(f"{'─'*70}")
print(f"{'AUC-ROC':<20} {test_results['auc_roc']:.4f}{'':10} {baseline_auc:.4f}{'':14} {test_results['auc_roc'] - baseline_auc:+.4f}")
print(f"{'Accuracy':<20} {test_results['accuracy']:.4f}{'':10} {baseline_acc:.4f}{'':14} {test_results['accuracy'] - baseline_acc:+.4f}")
print(f"{'Precision':<20} {test_results['precision']:.4f}{'':10} {baseline_metrics['resultados'][best_split]['precision']:.4f}{'':14} {test_results['precision'] - baseline_metrics['resultados'][best_split]['precision']:+.4f}")
print(f"{'Recall':<20} {test_results['recall']:.4f}{'':10} {baseline_metrics['resultados'][best_split]['recall']:.4f}{'':14} {test_results['recall'] - baseline_metrics['resultados'][best_split]['recall']:+.4f}")
print(f"{'F1-Score':<20} {test_results['f1']:.4f}{'':10} {baseline_metrics['resultados'][best_split]['f1_score']:.4f}{'':14} {test_results['f1'] - baseline_metrics['resultados'][best_split]['f1_score']:+.4f}")

print(f"\n💾 ARCHIVOS GENERADOS:")
output_files = [
    'modelo_rnn_final.h5',
    'training_history.png',
    'roc_curve_rnn.png',
    'confusion_matrix_rnn.png',
    'learning_curves.png',
    'reporte_rnn.txt',
    'metricas_rnn.json'
]

for file in output_files:
    file_path = ruta_deeplearning / file
    if file_path.exists():
        size = file_path.stat().st_size
        if size > 1024*1024:
            size_str = f"{size/(1024*1024):.2f} MB"
        elif size > 1024:
            size_str = f"{size/1024:.2f} KB"
        else:
            size_str = f"{size} B"
        print(f"  ✅ {file:<40} ({size_str})")
    else:
        print(f"  ❌ {file}")

print(f"\n📁 Directorio de salida: {ruta_deeplearning.absolute()}")

print(f"\n🎯 EVALUACIÓN COMPARATIVA:")
if test_results['auc_roc'] > baseline_auc:
    print(f"  ✅ MEJORA EN AUC-ROC: {test_results['auc_roc'] - baseline_auc:+.4f} ({((test_results['auc_roc'] - baseline_auc)/baseline_auc)*100:+.2f}%)")
else:
    print(f"  ⚠️  SIN MEJORA en AUC-ROC: {test_results['auc_roc'] - baseline_auc:+.4f}")

if test_results['accuracy'] > baseline_acc:
    print(f"  ✅ MEJORA EN ACCURACY: {test_results['accuracy'] - baseline_acc:+.4f} ({((test_results['accuracy'] - baseline_acc)/baseline_acc)*100:+.2f}%)")
else:
    print(f"  ⚠️  SIN MEJORA en ACCURACY: {test_results['accuracy'] - baseline_acc:+.4f}")

print(f"\n🎯 PRÓXIMOS PASOS:")
print(f"  1. Validar modelo en datos reales de producción")
print(f"  2. Comparar tiempo de inference entre DeepLearning y XGBoost")
print(f"  3. Si aún hay margen de mejora: considerar Fase 4 (Ensemble)")
print(f"  4. Implementar pipeline de producción con mejor modelo")
print(f"  5. Monitorear performance en tiempo real")

print(f"\n" + "="*80)


✅ FASE 3: DEEP LEARNING - COMPLETADO

📊 RESULTADOS FINALES - DEEP LEARNING (Fase 3):

Métricas en Test Set:
  • AUC-ROC:    0.9672
  • Accuracy:   0.8759
  • Precision:  0.3599
  • Recall:     0.9560
  • F1-Score:   0.5230

📊 COMPARATIVA: DEEPLEARNING vs XGBOOST BASELINE

Métrica              DeepLearning    XGBoost Baseline     Diferencia     
──────────────────────────────────────────────────────────────────────
AUC-ROC              0.9672           0.9852               -0.0180
Accuracy             0.8759           0.9464               -0.0705
Precision            0.3599           0.9171               -0.5572
Recall               0.9560           0.9813               -0.0253
F1-Score             0.5230           0.9482               -0.4252

💾 ARCHIVOS GENERADOS:
  ✅ modelo_rnn_final.h5                      (681.99 KB)
  ✅ training_history.png                     (71.04 KB)
  ✅ roc_curve_rnn.png                        (44.63 KB)
  ✅ confusion_matrix_rnn.png                 (36.40 KB